# Correcciones al notebook 02

Al analizar los resultados aparecen tres problemas. Dos son defectos del diseño
experimental y uno es un error de especificación del modelo.

| # | Problema | Sustituye a |
|---|---|---|
| **A** | Los 12 orígenes de validación caen **todos en miércoles**, porque el paso de 168 h es exactamente una semana. La evaluación no cubre fines de semana ni festivos ordinarios, y favorece artificialmente al modelo ingenuo diario. | Sección 3 |
| **B** | SARIMAX se especificó con `d=0` y sin término constante, de modo que revierte a la media del año de entrenamiento. Subestima 2.174 MW en octubre y **sobrestima** 1.918 MW en Navidad. | Sección 5.3 |
| **C** | La media aritmética del MAPE está dominada por una única ventana (25 de diciembre), donde todos los modelos fallan. | Sección 7 |

---

## Los resultados actuales, en contexto

| Modelo | MAPE con Navidad | MAPE sin Navidad | Mediana |
|---|---|---|---|
| Ingenuo diario | 2,94 % | 1,65 % | 1,32 % |
| SARIMA clásico | 4,54 % | 3,40 % | 2,89 % |
| Ingenuo semanal | 6,89 % | 3,82 % | 3,01 % |
| Prophet | 5,42 % | 4,51 % | 4,37 % |
| SARIMAX + Fourier | 7,16 % | 6,84 % | 6,94 % |
| Perfil semanal | 9,18 % | 7,43 % | 7,31 % |

La ventana del 25 de diciembre eleva el MAPE del ingenuo semanal de 3,82 % a
6,89 %: casi duplica la métrica ella sola. En esa ventana el error de ese modelo
alcanza el 40,68 %, lo que resulta lógico —compara Navidad con un miércoles
laborable ordinario— pero distorsiona la comparación global.

---
## Celda A — Orígenes de validación que cubran toda la semana

**Sustituye a la sección 3.**

Con `PASO = 168` todos los orígenes caen en el mismo día de la semana. El efecto
es doble: la evaluación ignora sábados y domingos, y el modelo ingenuo diario
resulta beneficiado, porque predice miércoles a partir de martes, ambos días
laborables de perfil casi idéntico.

La solución es emplear un paso que no sea múltiplo de 7 días. Con **5 días** y
14 orígenes se obtienen exactamente dos de cada día de la semana.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# A.1 — Parámetros corregidos
# ═══════════════════════════════════════════════════════════════════════════
HORIZONTE  = 24
N_VENTANAS = 14
PASO       = 120     # 5 días: rota por todos los días de la semana

VENTANAS = generar_ventanas(len(y), HORIZONTE, N_VENTANAS, PASO)

# ── Verificación: ¿se cubren todos los días de la semana? ────────────────
dias = pd.Series([y.index[a].day_name() for a, _ in VENTANAS])
reparto = dias.value_counts()

print(f"Orígenes generados: {len(VENTANAS)}")
print(f"Días de la semana distintos: {reparto.nunique() if len(reparto) else 0} de 7\n")
print(reparto.to_string())

if len(reparto) < 7:
    print("\nATENCIÓN: la evaluación no cubre todos los días de la semana.")
    print("Ajusta PASO a un valor que no sea múltiplo de 168.")
else:
    print("\nCobertura semanal completa.")

periodo = (y.index[VENTANAS[-1][1]-1] - y.index[VENTANAS[0][0]]).days
print(f"Periodo evaluado: {periodo} días "
      f"({y.index[VENTANAS[0][0]]:%d-%b-%Y} a {y.index[VENTANAS[-1][1]-1]:%d-%b-%Y})")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# A.2 — Etiquetar cada ventana con su contexto de calendario
# ═══════════════════════════════════════════════════════════════════════════
# Permite separar después los días ordinarios de los festivos.

contexto = []
for i, (a, b) in enumerate(VENTANAS):
    tramo = df.iloc[a:b]
    fecha = y.index[a]
    contexto.append({
        "ventana":  i,
        "fecha":    fecha.date(),
        "dia":      fecha.day_name(),
        "festivo":  int(tramo["festivo"].max()) if "festivo" in df.columns else 0,
        "finde":    int(fecha.dayofweek >= 5),
        "demanda_media": tramo["demanda_mw"].mean(),
    })

CONTEXTO = pd.DataFrame(contexto).set_index("ventana")
CONTEXTO["tipo"] = np.select(
    [CONTEXTO["festivo"] == 1, CONTEXTO["finde"] == 1],
    ["Festivo", "Fin de semana"],
    default="Laborable")

display(CONTEXTO)
print(CONTEXTO["tipo"].value_counts().to_string())

---
## Celda B — Corrección de la especificación de SARIMAX

**Sustituye a la sección 5.3.**

El modelo se especificó como `SARIMAX(orden=(2,0,1), exog=Fourier)`. Con `d=0` y
sin término constante, `statsmodels` asume media cero, de modo que el nivel de
la serie debe reconstruirse únicamente a partir de la parte autorregresiva. El
resultado es una fuerte reversión a la media del periodo de entrenamiento:

| Ventana | Real | Predicho | Sesgo |
|---|---|---|---|
| Octubre | 28.421 MW | 26.247 MW | −2.174 |
| Noviembre | 29.869 MW | 27.802 MW | −2.067 |
| Navidad | 23.032 MW | 24.950 MW | **+1.918** |

El patrón es inequívoco: predice siempre algo próximo a la media anual, por
debajo cuando la demanda es alta y por encima cuando es baja.

**La corrección es aplicar una diferenciación de primer orden (`d=1`).** Así el
modelo trabaja sobre los incrementos y se adapta al nivel local en lugar de
anclarse a la media global. En pruebas controladas esto reduce el sesgo de
−1.388 MW a −76 MW.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# B.1 — SARIMAX corregido
# ═══════════════════════════════════════════════════════════════════════════
ORDEN_ARIMA = (2, 1, 1)      # d=1: diferenciación de primer orden
MAX_PUNTOS  = 8760

# Se añade el ciclo anual a los términos de Fourier, que antes solo cubrían
# la estacionalidad diaria y semanal.
PERIODOS = {24: 4, 168: 2, 8766: 2}


def pred_sarimax_fourier(train, h, devolver_modelo=False):
    """SARIMAX con diferenciación y estacionalidad múltiple vía Fourier."""
    sub = train.iloc[-MAX_PUNTOS:]
    n = len(sub)

    exog_train = terminos_fourier(n, PERIODOS, offset=0).values
    exog_test  = terminos_fourier(h, PERIODOS, offset=n).values

    modelo = SARIMAX(sub.values, exog=exog_train, order=ORDEN_ARIMA,
                     enforce_stationarity=False, enforce_invertibility=False
                     ).fit(disp=False, maxiter=50)

    pred = modelo.forecast(steps=h, exog=exog_test)
    return (pred, modelo) if devolver_modelo else pred


# ── Comprobación del sesgo sobre los últimos orígenes ────────────────────
print("Verificación del sesgo tras la corrección\n")
sesgos = []
for a, b in VENTANAS[-4:]:
    tr, te = y.iloc[:a], y.iloc[a:b]
    p = pred_sarimax_fourier(tr, len(te))
    s = p.mean() - te.values.mean()
    sesgos.append(s)
    print(f"  {y.index[a]:%d-%b}: real {te.mean():>8,.0f} | "
          f"pred {p.mean():>8,.0f} | sesgo {s:+7,.0f} MW")

print(f"\nSesgo medio: {np.mean(sesgos):+,.0f} MW")
if abs(np.mean(sesgos)) < 500:
    print("Sesgo corregido.")
else:
    print("Sigue habiendo sesgo. Prueba a reducir MAX_PUNTOS a 24*90.")

---
## Celda C — Métricas robustas y separación por tipo de día

**Sustituye a la sección 7 (tabla de resultados).**

La media aritmética del MAPE resulta muy sensible a una única ventana atípica.
Se reportan tres medidas complementarias:

- **Media** — comparable con la literatura, pero sensible a valores extremos.
- **Mediana** — refleja el comportamiento en un día típico.
- **Desviación típica** — mide la estabilidad entre orígenes.

Además se desglosa el error según el tipo de día, lo que permite comprobar si un
modelo falla de forma sistemática en fines de semana o festivos.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# C.1 — Tabla de resultados con métricas robustas
# ═══════════════════════════════════════════════════════════════════════════
filas = []

for modelo, p in predicciones.items():
    p = p.copy()
    p["ape"] = np.abs((p["real"] - p["pred"]) / p["real"]) * 100
    por_ventana = p.groupby("ventana")["ape"].mean()

    filas.append({
        "Modelo":      modelo,
        "MAPE medio":  por_ventana.mean(),
        "MAPE mediana": por_ventana.median(),
        "MAPE std":    por_ventana.std(),
        "MAPE peor":   por_ventana.max(),
        "MAE":         np.abs(p["real"] - p["pred"]).mean(),
        "RMSE":        np.sqrt(((p["real"] - p["pred"]) ** 2).mean()),
    })

tabla = pd.DataFrame(filas).set_index("Modelo").sort_values("MAPE mediana")

ingenuos = [m for m in tabla.index if m.startswith(("Ingenuo", "Perfil"))]
referencia = tabla.loc[ingenuos, "MAPE mediana"].min()
tabla["vs ingenuo"] = (referencia - tabla["MAPE mediana"]) / referencia * 100

print(f"Referencia (mejor ingenuo, mediana): {referencia:.2f} %\n")
display(tabla.round(2))

tabla.to_csv(DIR_RESULTADOS / "comparativa_baseline.csv")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# C.2 — Error según el tipo de día
# ═══════════════════════════════════════════════════════════════════════════
desglose = []

for modelo, p in predicciones.items():
    p = p.copy()
    p["ape"] = np.abs((p["real"] - p["pred"]) / p["real"]) * 100
    por_v = p.groupby("ventana")["ape"].mean().rename("ape")
    unido = pd.concat([por_v, CONTEXTO["tipo"]], axis=1).dropna()

    fila = {"Modelo": modelo}
    for tipo in ["Laborable", "Fin de semana", "Festivo"]:
        sel = unido[unido["tipo"] == tipo]["ape"]
        fila[tipo] = sel.mean() if len(sel) else np.nan
    desglose.append(fila)

por_tipo = pd.DataFrame(desglose).set_index("Modelo")
por_tipo = por_tipo.loc[tabla.index]

print("MAPE medio según el tipo de día\n")
display(por_tipo.round(2))

# ── Gráfico ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

colores = ["#888780" if m.startswith(("Ingenuo", "Perfil")) else "#4C72B0"
           for m in tabla.index]

axes[0].barh(tabla.index, tabla["MAPE mediana"], color=colores,
             xerr=tabla["MAPE std"], error_kw={"lw": 1, "alpha": .5})
axes[0].axvline(referencia, color="crimson", ls="--", lw=1.2,
                label=f"Mejor ingenuo ({referencia:.2f} %)")
axes[0].set_xlabel("MAPE mediana (%)")
axes[0].set_title("Error mediano con dispersión", fontweight="bold")
axes[0].legend(fontsize=8)
axes[0].invert_yaxis()

por_tipo.plot(kind="barh", ax=axes[1], width=0.75,
              color=["#55A868", "#DD8452", "#C44E52"])
axes[1].set_xlabel("MAPE (%)")
axes[1].set_title("Error según el tipo de día", fontweight="bold")
axes[1].legend(fontsize=8, title=None)
axes[1].invert_yaxis()

plt.tight_layout()
guardar_figura("11_comparativa_baseline")
plt.show()

> **Cómo interpretarlo.** Si un modelo presenta un error muy superior en la
> columna *Festivo*, no está aprovechando la información de calendario. Es
> justamente lo que se espera de los modelos ingenuos, y es la carencia que los
> modelos de aprendizaje automático del notebook 03 deberían corregir, ya que
> disponen de `festivo`, `vispera_festivo` y `laborable` como variables.

---
## Qué esperar tras volver a ejecutar

**El modelo ingenuo diario perderá ventaja.** Con orígenes en sábado y domingo
dejará de predecir siempre un laborable a partir de otro laborable. Es probable
que el ingenuo semanal pase a ser la mejor línea base, que es el resultado
habitual en demanda eléctrica.

**SARIMAX debería situarse por debajo del 4 %.** Si tras la corrección sigue
por encima del 6 %, prueba a reducir `MAX_PUNTOS` a `24*90`: para un horizonte
de 24 horas el nivel reciente pesa más que un año completo de historia.

**Prophet tardó 29 minutos en tu ejecución.** Con 14 ventanas será algo más.
Para acelerarlo puedes reducir el entrenamiento a los dos últimos años
(`train.iloc[-17520:]`) dentro de `pred_prophet`, o dejarlo corriendo mientras
haces otra cosa.

---

## Sobre la ventana de Navidad

No la elimines. Es el hallazgo más interesante de esta tanda: **todos** los
modelos fallan el 25 de diciembre, con errores de entre el 10 % y el 40 %. Eso
conecta directamente con el análisis de anomalías del notebook 01, donde el
70 % de las desviaciones se concentraba en festivos.

Para la memoria es un argumento de peso: los modelos de línea base carecen de
información de calendario, y ahí es donde los modelos de aprendizaje automático
tienen margen de mejora. Preséntalo como resultado, no como un problema.